# Offline Pairwise Probability Histogram Animation

This notebook loads saved run artifacts and creates an animation of a 1D histogram over a pairwise probability axis.

For each sample in each snapshot:
- left end: probability of the target class (`class_index`)
- right end: probability of the strongest competing class (`argmax` over all classes except `class_index`)

The pair is normalized to sum to 1 using one of:
- `conditional` (standard): divide by pair sum
- `softmax`: apply softmax to the two values
- `scaled`: multiply by class-specific scales, then renormalize

In [ ]:
from pathlib import Path
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML

In [ ]:
def load_run_payload(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return torch.load(path, map_location='cpu', weights_only=False)


def get_ts(bs=320, epochs=200, save_init=5, save_freq=4, len_data=50000):
    t = 0
    ts = [t]
    n_batches = len_data // bs
    for epoch in range(epochs):
        for i in range(n_batches):
            t += 1
            if epoch < save_init and i % (n_batches // save_freq) == 0:
                ts.append(t)

        if save_init <= epoch <= 25:
            ts.append(t)
        elif 25 < epoch <= 65 and epoch % 4 == 0:
            ts.append(t)
        elif epoch > 65 and epoch % 15 == 0 or (epoch == epochs - 1):
            ts.append(t)
    return ts


def extract_snapshots(payload):
    if isinstance(payload, dict) and 'data' in payload:
        snapshots = payload['data']
    else:
        snapshots = payload
    if not isinstance(snapshots, (list, tuple)):
        raise ValueError('Could not find snapshot list in run artifact.')
    if len(snapshots) == 0:
        raise ValueError('Snapshot list is empty.')
    return list(snapshots)


def _to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _softmax_numpy(x, axis=1):
    z = x - np.max(x, axis=axis, keepdims=True)
    ez = np.exp(z)
    return ez / np.clip(ez.sum(axis=axis, keepdims=True), 1e-12, None)


def snapshot_to_probs(snapshot, split_key='yh', input_kind='log_probs'):
    if split_key not in snapshot:
        raise KeyError(f"Snapshot missing key '{split_key}'. Available: {list(snapshot.keys())}")

    arr = _to_numpy(snapshot[split_key])
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D array, got shape {arr.shape}')

    kind = str(input_kind).lower()
    if kind == 'log_probs':
        probs = np.exp(arr)
    elif kind == 'logits':
        probs = _softmax_numpy(arr, axis=1)
    elif kind == 'probs':
        probs = np.asarray(arr, dtype=float)
    else:
        raise ValueError("input_kind must be one of {'log_probs','logits','probs'}")

    if probs.shape[1] < 2:
        raise ValueError(f'Need at least 2 classes, got {probs.shape[1]}')
    return probs

In [ ]:
def pairwise_correct_vs_best_other(probs, class_index):
    c = probs.shape[1]
    cls = int(class_index)
    if cls < 0 or cls >= c:
        raise ValueError(f'class_index={cls} out of range for {c} classes')

    p_correct = probs[:, cls]
    mask = np.ones(c, dtype=bool)
    mask[cls] = False
    p_best_other = probs[:, mask].max(axis=1)
    return p_correct, p_best_other


def normalize_pair(p_correct, p_best_other, mode='conditional', scale=(1.0, 1.0), temperature=1.0):
    eps = 1e-12
    pc = np.asarray(p_correct, dtype=float)
    po = np.asarray(p_best_other, dtype=float)
    m = str(mode).lower()

    if m == 'conditional':
        # Standard conditional renormalization over the two-class subset.
        z = np.clip(pc + po, eps, None)
        pcn = pc / z
        pon = po / z
    elif m == 'softmax':
        pair = np.stack([pc, po], axis=1)
        t = max(float(temperature), eps)
        z = pair / t
        z = z - np.max(z, axis=1, keepdims=True)
        ez = np.exp(z)
        denom = np.clip(ez.sum(axis=1, keepdims=True), eps, None)
        out = ez / denom
        pcn, pon = out[:, 0], out[:, 1]
    elif m == 'scaled':
        # Direct multiplicative scaling before renormalization.
        a, b = float(scale[0]), float(scale[1])
        spc = np.clip(a * pc, 0.0, None)
        spo = np.clip(b * po, 0.0, None)
        z = np.clip(spc + spo, eps, None)
        pcn = spc / z
        pon = spo / z
    else:
        raise ValueError("mode must be one of {'conditional','softmax','scaled'}")

    return pcn, pon


def build_pair_series(
    snapshots,
    class_index,
    split_key='yh',
    input_kind='log_probs',
    normalize_mode='conditional',
    scale=(1.0, 1.0),
    temperature=1.0,
):
    series = []
    for s in snapshots:
        probs = snapshot_to_probs(s, split_key=split_key, input_kind=input_kind)
        pc, po = pairwise_correct_vs_best_other(probs, class_index=class_index)
        pcn, pon = normalize_pair(pc, po, mode=normalize_mode, scale=scale, temperature=temperature)
        series.append((pcn, pon))
    return series

In [ ]:
def make_pair_hist_animation(
    snapshots,
    ts,
    title,
    class_index,
    split_key='yh',
    input_kind='log_probs',
    normalize_mode='conditional',
    scale=(1.0, 1.0),
    temperature=1.0,
    bins=40,
    interval=200,
    save_mp4=False,
    out_name='pair_hist_offline',
    color='#1f77b4'
):
    pair_series = build_pair_series(
        snapshots=snapshots,
        class_index=class_index,
        split_key=split_key,
        input_kind=input_kind,
        normalize_mode=normalize_mode,
        scale=scale,
        temperature=temperature,
    )

    if len(pair_series) == 0:
        raise ValueError('No frames to animate')

    # x-axis is normalized correct-class probability; best-other is 1-x.
    x0 = pair_series[0][0]
    bin_edges = np.linspace(0.0, 1.0, int(bins) + 1)
    counts0, _ = np.histogram(x0, bins=bin_edges)

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    widths = np.diff(bin_edges)
    bars = ax.bar(bin_edges[:-1], counts0, width=widths, align='edge', color=color, alpha=0.9, edgecolor='white', linewidth=0.5)

    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel('Normalized mass on class_index (best-other is 1 - x)')
    ax.set_ylabel('Count')
    ax.set_yscale('log')
    ax.grid(axis='y', alpha=0.25)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.0, alpha=0.6)
    ax.text(0.0, 1.02, 'best-other end', transform=ax.transAxes, ha='left', va='bottom', fontsize=9)
    ax.text(1.0, 1.02, 'correct-class end', transform=ax.transAxes, ha='right', va='bottom', fontsize=9)

    ymax = max(1, int(max(np.histogram(v[0], bins=bin_edges)[0].max() for v in pair_series) * 1.1))
    ax.set_ylim(0, ymax)

    mode_txt = f'mode={normalize_mode}'
    if str(normalize_mode).lower() == 'scaled':
        mode_txt += f', scale={tuple(scale)}'
    if str(normalize_mode).lower() == 'softmax':
        mode_txt += f', temp={float(temperature):.3g}'

    def update(i):
        x = pair_series[i][0]
        counts, _ = np.histogram(x, bins=bin_edges)
        for bar, h in zip(bars, counts):
            bar.set_height(h)
        step = ts[i] if i < len(ts) else i
        ax.set_title(f'{title} | class_index={class_index} | {mode_txt} | step {step}')
        return tuple(bars)

    ani = FuncAnimation(fig, update, frames=len(pair_series), interval=interval, blit=False)

    if save_mp4:
        out_path = Path(out_name).with_suffix('.mp4')
        out_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            import imageio_ffmpeg as iioff
            plt.rcParams['animation.ffmpeg_path'] = iioff.get_ffmpeg_exe()
        except Exception:
            pass
        writer = FFMpegWriter(fps=max(1, int(1000 / max(interval, 1))), bitrate=1800)
        ani.save(str(out_path), writer=writer, dpi=140)
        print('Saved', out_path.resolve())

    return fig, ani

In [ ]:
dataset = 'CIFAR10'
class_index = 1
normalization_mode = 'conditional'  # 'conditional' | 'softmax' | 'scaled'
scale = (1.0, 1.0)                # used only when normalization_mode='scaled'
temperature = 1.0                 # used only when normalization_mode='softmax'

for method in ['Bayesian', 'Full', 'RhoLoss', 'DivBS', 'GradNorm', 'Uniform']:
    run_path = (
        './results/' + str(dataset) +
        '/{\"bsel\":\"' + str(method) +
        '\",\"seed\":16,\"model\":\"ResNet_torchvision\",\"opt\":\"adam\",\"bs\":320,\"ratio\":0.1,\"lr\":0.001,\"wd\":0.0}.p'
    )

    payload = load_run_payload(run_path)
    snapshots = extract_snapshots(payload)
    print(method, 'snapshots:', len(snapshots))

    ts = get_ts()
    fig, ani = make_pair_hist_animation(
        snapshots=snapshots,
        ts=ts,
        title=f'{dataset} {method}',
        class_index=class_index,
        split_key='yh',
        input_kind='log_probs',
        normalize_mode=normalization_mode,
        scale=scale,
        temperature=temperature,
        bins=100,
        interval=200,
        save_mp4=True,
        out_name=f'./runs/hex_frames/{dataset}/{method}_pair_hist_{normalization_mode}'
    )
    display(HTML(ani.to_jshtml()))
    plt.close(fig)